# Demo

In [6]:
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is not set. Add it to your environment or .env file.")

OPENAI_MODEL = os.getenv("OPENAI_MODEL")

In [13]:
import json
import sys
from pathlib import Path

# Ensure src/ is importable whether cwd is repo root or notebooks/
cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(repo_root / "src"))

from openclam.agents.market_technical.market_technical_agent import (
    compute_all_indicators,
    create_market_llm,
    run_market_analysis,
)

In [8]:
llm = create_market_llm(
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    temperature=0.2,
)

In [9]:
# No time frame
result = run_market_analysis(
    "Analyze AAPL market reaction after earnings",
    llm=llm,
)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "ticker": "AAPL",
  "company": "Apple Inc.",
  "agent_name": "Market & Technical Analysis Agent",
  "short_term_stance": "bullish",
  "long_term_stance": "bullish",
  "confidence": 0.82,
  "confidence_rationale": "Strong short-term momentum signals (positive returns, RSI 66, bullish MACD crossover) combined with price trading above MA20/MA50 and a volume spike support continuation. Market regime is bullish_trend. However, relative underperformance vs SPY and earnings-driven volatility introduce some risk to sustained upside.",
  "key_signals": [
    {
      "type": "Returns",
      "signal": "positive short-term momentum",
      "evidence": "Mean of recent returns: 0.37%"
    },
    {
      "type": "RSI",
      "signal": "bullish momentum",
      "evidence": "RSI at 66.36"
    },
    {
      "type": "Moving Averages",
      "signal": "bullish trend alignment",
      "evidence": "Price 280.14, MA20 266.36, MA50 261.22"
    },
    {
      "type": "MACD",
      "signal": "bullish cros

In [10]:
# Explicit date
result = run_market_analysis(
    "Will NVDA be bullish on 2026/4/3?",
    llm=llm,
)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "ticker": "NVDA",
  "company": "NVIDIA Corporation",
  "agent_name": "Market & Technical Analysis Agent",
  "short_term_stance": "neutral",
  "long_term_stance": "bullish",
  "confidence": 0.68,
  "confidence_rationale": "Longer-term setup shows bullish trend signals (price above MA20/MA50/MA200, bullish market regime, and strong relative strength). Near-term momentum is mixed with negative returns, a bearish MACD crossover, and weak volume, indicating potential near-term consolidation despite the bullish longer-term context.",
  "key_signals": [
    {
      "type": "Returns",
      "signal": "negative short-term momentum",
      "evidence": "Mean of recent returns: -0.13%"
    },
    {
      "type": "RSI",
      "signal": "bullish momentum",
      "evidence": "RSI at 52.98"
    },
    {
      "type": "Moving Averages",
      "signal": "bullish trend alignment",
      "evidence": "Price 198.45, MA20 197.22, MA50 187.15, MA200 183.84"
    },
    {
      "type": "MACD",
      "signal

# Evaluation

In [25]:
import importlib
import sys
import pandas as pd

# Remove cached module if it exists
module_name = 'openclam.agents.market_technical.market_technical_agent'
if module_name in sys.modules:
    del sys.modules[module_name]

# Now import the updated functions
from openclam.agents.market_technical.market_technical_agent import (
    build_earnings_price_eval,
    run_agent_event_window_eval,
    summarize_eval_results,
)

print("✓ Successfully imported evaluation functions")

# Sample earnings data (MAG7 + others)
earnings_data = pd.DataFrame({
    'ticker': ['TSLA', 'META', 'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA'],
    'company': ['Tesla', 'Meta Platforms', 'Apple', 'Microsoft', 'Alphabet', 'Amazon', 'Nvidia'],
    'earnings_date': ['2026-01-28', '2026-01-28', '2026-01-29', '2026-01-28', '2026-02-03', '2026-02-05', '2026-02-25']
})

print("\nEarnings Data:")
print(earnings_data)


✓ Successfully imported evaluation functions

Earnings Data:
  ticker         company earnings_date
0   TSLA           Tesla    2026-01-28
1   META  Meta Platforms    2026-01-28
2   AAPL           Apple    2026-01-29
3   MSFT       Microsoft    2026-01-28
4  GOOGL        Alphabet    2026-02-03
5   AMZN          Amazon    2026-02-05
6   NVDA          Nvidia    2026-02-25


In [17]:
# Build event-window price paths around earnings dates
print("Building earnings price evaluation dataset...")
summary_df, paths_df = build_earnings_price_eval(
    earnings_df=earnings_data,
    pre_trading_days=7,
    post_trading_days=7,
    long_post_trading_days=30,
    benchmarks=("SPY", "QQQ"),
)

print(f"\nSummary DataFrame Shape: {summary_df.shape}")
print("\nFirst few rows:")
print(summary_df[['ticker', 'company', 'earnings_date', 'post_7d_return', 'abnormal_vs_qqq']].head(10))


Building earnings price evaluation dataset...

Summary DataFrame Shape: (7, 16)

First few rows:
  ticker         company earnings_date  post_7d_return  abnormal_vs_qqq
0   TSLA           Tesla    2026-01-28       -0.071977        -0.047012
1   META  Meta Platforms    2026-01-28        0.078581         0.103547
2   AAPL           Apple    2026-01-29        0.118403         0.136542
3   MSFT       Microsoft    2026-01-28       -0.099313        -0.074347
4  GOOGL        Alphabet    2026-02-03       -0.028512        -0.010446
5   AMZN          Amazon    2026-02-05       -0.184241        -0.135930
6   NVDA          Nvidia    2026-02-25        0.001258        -0.013412


In [22]:
# Run market technical agent evaluation on event windows
print("Running market technical agent on event windows...")
print("This will analyze each stock's technical situation around earnings dates.\n")

agent_eval, agent_reports = run_agent_event_window_eval(
    summary_df,
    llm=llm,
    lookback_days=14,
    neutral_band=0.02,
    long_post_trading_days=30,
    quiet=True,
)

print(f"✓ Evaluation completed for {len(agent_eval)} cases\n")

Running market technical agent on event windows...
This will analyze each stock's technical situation around earnings dates.

✓ Evaluation completed for 7 cases



In [26]:
# Display comprehensive results
display_cols = [
    'ticker', 'company', 'earnings_date',
    'post_7d_return', 'abnormal_vs_qqq', 'realized_direction_vs_qqq',
    'post_30d_return', 'abnormal_30d_vs_qqq', 'realized_30d_direction_vs_qqq',
    'agent_short_term_stance', 'agent_long_term_stance', 'agent_stance',
    'agent_confidence', 'confidence_rationale',
    'short_direction_match', 'short_direction_match_reason',
    'long_direction_match', 'long_direction_match_reason',
    'direction_match', 'direction_match_reason'
]

# Filter to only columns that exist
available_display_cols = [col for col in display_cols if col in agent_eval.columns]

print("Agent Evaluation Results - Full Output:")
print(f"Shape: {agent_eval.shape}")
print("\nAvailable columns:", agent_eval.columns.tolist())
print("\nData preview:")
print(agent_eval[available_display_cols].to_string(max_rows=5, max_colwidth=50))


Agent Evaluation Results - Full Output:
Shape: (7, 18)

Available columns: ['ticker', 'company', 'earnings_date', 'post_7d_return', 'abnormal_vs_qqq', 'realized_direction_vs_qqq', 'post_30d_return', 'abnormal_30d_vs_qqq', 'realized_30d_direction_vs_qqq', 'news_context_ready', 'report_ready', 'agent_short_term_stance', 'agent_long_term_stance', 'agent_confidence', 'short_direction_match', 'short_direction_match_reason', 'long_direction_match', 'long_direction_match_reason']

Data preview:
   ticker         company earnings_date  post_7d_return  abnormal_vs_qqq realized_direction_vs_qqq  post_30d_return  abnormal_30d_vs_qqq realized_30d_direction_vs_qqq agent_short_term_stance agent_long_term_stance agent_confidence short_direction_match              short_direction_match_reason long_direction_match              long_direction_match_reason
0    TSLA           Tesla    2026-01-28       -0.071977        -0.047012                      down        -0.079977            -0.057490              

In [19]:
# Summarize evaluation results
print("Evaluating prediction accuracy...\n")
eval_summary = summarize_eval_results(agent_eval)

print("Short-Term (7-day) Analysis:")
print(f"  - Evaluable cases: {eval_summary['short_evaluable_cases']}")
print(f"  - Correct predictions: {eval_summary['short_matched_cases']}")
print(f"  - Missed predictions: {eval_summary['short_missed_cases']}")
print(f"  - Accuracy: {eval_summary['short_accuracy']:.2%}" if eval_summary['short_accuracy'] else "  - Accuracy: N/A")
print(f"  - Neutral stance rate: {eval_summary['short_neutral_rate']:.2%}" if eval_summary['short_neutral_rate'] else "  - Neutral rate: N/A")

print("\nLong-Term (30-day) Analysis:")
print(f"  - Evaluable cases: {eval_summary['long_evaluable_cases']}")
print(f"  - Correct predictions: {eval_summary['long_matched_cases']}")
print(f"  - Missed predictions: {eval_summary['long_missed_cases']}")
print(f"  - Accuracy: {eval_summary['long_accuracy']:.2%}" if eval_summary['long_accuracy'] else "  - Accuracy: N/A")
print(f"  - Neutral stance rate: {eval_summary['long_neutral_rate']:.2%}" if eval_summary['long_neutral_rate'] else "  - Neutral rate: N/A")

print("\n" + "="*60)
print("Agent Reports Sample:")
for ticker in list(agent_reports.keys())[:2]:
    report = agent_reports[ticker]
    print(f"\n{ticker}: {report.get('company', ticker)}")
    print(f"  Short-term stance: {report.get('short_term_stance', 'N/A')}")
    print(f"  Long-term stance: {report.get('long_term_stance', 'N/A')}")
    print(f"  Confidence: {report.get('confidence', 'N/A'):.2f}")
    print(f"  Core insight: {report.get('core_insight', 'N/A')[:100]}...")


Evaluating prediction accuracy...

Short-Term (7-day) Analysis:
  - Evaluable cases: 3
  - Correct predictions: 1
  - Missed predictions: 2
  - Accuracy: 33.33%
  - Neutral stance rate: 42.86%

Long-Term (30-day) Analysis:
  - Evaluable cases: 4
  - Correct predictions: 1
  - Missed predictions: 3
  - Accuracy: 25.00%
  - Neutral stance rate: 28.57%

Agent Reports Sample:

TSLA: Tesla, Inc.
  Short-term stance: neutral
  Long-term stance: neutral
  Confidence: 0.62
  Core insight: MACD bullish crossover with price above key moving averages suggests potential near-term continuatio...

META: Meta Platforms
  Short-term stance: bearish
  Long-term stance: neutral
  Confidence: 0.78
  Core insight: Multiple momentum and relative strength indicators point to near-term downside pressure, but the ran...
